<a href="https://colab.research.google.com/github/KchandraSwaroop/AI-lab/blob/main/1WN24CS128_Week8_FOL_ForwardReasoning.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [5]:
# Forward Chaining
def is_var(x):
    return isinstance(x, str) and x.startswith("?")

def subst(x, theta):
    while is_var(x) and x in theta:
        x = theta[x]
    return x

def unify(a, b, theta):
    if is_var(a):
        theta = theta.copy()
        theta[a] = b
        return theta
    if is_var(b):
        theta = theta.copy()
        theta[b] = a
        return theta
    if a == b:
        return theta
    return None

def match(p, f, theta):
    if p[0] != f[0]:
        return None

    for x, y in zip(p[1:], f[1:]):
        theta = unify(subst(x, theta), subst(y, theta), theta)
        if theta is None:
            return None

    return theta

def forward_chaining(facts, rules, goal):

    while True:
        new = []

        for conditions, result in rules:
            substitutions = [{}]

            for condition in conditions:
                temp = []

                for theta in substitutions:
                    for fact in facts:
                        t = match(
                            tuple(subst(x, theta) for x in condition),
                            fact, theta
                        )
                        if t is not None:
                            temp.append(t)

                substitutions = temp

            for theta in substitutions:
                fact = tuple(subst(x, theta) for x in result)

                if fact not in facts and fact not in new:
                    new.append(fact)

        if not new:
            break

        facts.extend(new)
        print("New Facts:", new)

        if goal in facts:
            return True

    return goal in facts


# Initial Facts
facts = [
    ("Parent", "Alice", "Bob"),
    ("Parent", "Bob", "Charlie")
]

# Rules
rules = [
    ([("Parent", "?x", "?y")],
     ("Ancestor", "?x", "?y")),

    ([("Parent", "?x", "?y"),
      ("Ancestor", "?y", "?z")],
     ("Ancestor", "?x", "?z"))
]

# Goal
goal = ("Ancestor", "Alice", "Charlie")
print ("K CHANDRA SWAROOP | 1WN24CS128")

print("Initial Facts:", facts)
print("Goal:", goal)

if forward_chaining(facts, rules, goal):
    print("Goal is TRUE")
else:
    print("Goal is FALSE")

K CHANDRA SWAROOP | 1WN24CS128
Initial Facts: [('Parent', 'Alice', 'Bob'), ('Parent', 'Bob', 'Charlie')]
Goal: ('Ancestor', 'Alice', 'Charlie')
New Facts: [('Ancestor', 'Alice', 'Bob'), ('Ancestor', 'Bob', 'Charlie')]
New Facts: [('Ancestor', 'Alice', 'Charlie')]
Goal is TRUE


In [8]:
# GENERAL FOL FORWARD CHAINING ALGORITHM
# Parse atomic sentence
def parse_atom(s):
    name, args = s.strip().split("(")
    args = args.rstrip(")")
    return (name.strip(),
            tuple(x.strip() for x in args.split(",")))
# Parse FOL rule
def parse_rule(s):
    left, right = s.split("->")
    premises = [parse_atom(x) for x in left.split("&")]
    conclusion = parse_atom(right)
    return premises, conclusion
# Check variable
def is_variable(x):
    return isinstance(x, str) and x.startswith("?")
# Apply substitution
def subst(theta, atom):
    return (atom[0],
            tuple(theta.get(x, x) for x in atom[1]))
# Unification
def unify(x, y, theta=None):
    if theta is None:
        theta = {}

    x = subst(theta, x)
    y = subst(theta, y)
    if x == y:
        return theta
    if x[0] != y[0] or len(x[1]) != len(y[1]):
        return None
    for a, b in zip(x[1], y[1]):
        a = theta.get(a, a)
        b = theta.get(b, b)
        if a == b:
            continue
        if is_variable(a):
            theta[a] = b
        elif is_variable(b):
            theta[b] = a
        else:
            return None
    return theta
# Standardize variables
counter = 0
def standardize_variables(rule):
    global counter
    counter += 1
    premises, conclusion = rule
    variables = {}
    def rename(atom):
        args = []
        for arg in atom[1]:
            if is_variable(arg):
                if arg not in variables:
                    variables[arg] = arg + "_" + str(counter)
                args.append(variables[arg])
            else:
                args.append(arg)
        return (atom[0], tuple(args))
    return ([rename(p) for p in premises], rename(conclusion))
# Match rule premises with KB facts
def match_premises(premises, facts, theta=None):
    if theta is None:
        theta = {}
    if not premises:
        return [theta]
    results = []
    for fact in facts:
        new_theta = unify(premises[0], fact, theta.copy())

        if new_theta is not None:
            results.extend(
                match_premises(
                    premises[1:], facts, new_theta
                )
            )

    return results
# FOL Forward Chaining Algorithm
def FOL_FC_ASK(KB, rules, query):

    while True:
        new = set()

        for rule in rules:

            # Standardize variables
            premises, conclusion = standardize_variables(rule)

            # Find matching substitutions
            substitutions = match_premises(premises, KB)

            for theta in substitutions:

                # Apply substitution
                q = subst(theta, conclusion)

                # Avoid duplicate facts
                if q not in KB and q not in new:
                    new.add(q)

                # Unify inferred sentence with query
                phi = unify(q, query, {})

                if phi is not None:
                    return phi

        # Stop if no new facts
        if not new:
            return None

        # Add new facts to KB
        KB.update(new)


# Display atomic sentence
def display(atom):
    return atom[0] + "(" + ", ".join(atom[1]) + ")"


# MAIN PROGRAM

KB = set()
rules = []

n = int(input("Enter number of facts: "))

print("Enter facts:")

for i in range(n):
    KB.add(parse_atom(input()))


r = int(input("Enter number of rules: "))

print("Enter rules:")

for i in range(r):
    rules.append(parse_rule(input()))


query = parse_atom(input("Enter query: "))

print("\nInitial Knowledge Base:")

for fact in KB:
    print(display(fact))


print("\nApplying Forward Chaining...")

answer = FOL_FC_ASK(KB, rules, query)

if answer is not None:
    print("\nQuery Proved:", display(query))
    print("Substitution:", answer)
    print("Result: TRUE")
else:
    print("\nQuery Cannot Be Proved:", display(query))
    print("Result: FALSE")


Enter number of facts: 1
Enter facts:
Barks(Tommy)
Enter number of rules: 2
Enter rules:
Barks(?x) -> Dog(?x)
Dog(?x) -> Mammal(?x)
Enter query: Mammal(Tommy)

Initial Knowledge Base:
Barks(Tommy)

Applying Forward Chaining...

Query Proved: Mammal(Tommy)
Substitution: {}
Result: TRUE
